# status

> Feedback, loading, and empty states

In [ ]:
#| default_exp status

An interface needs to explain what happened, what is still happening, and why there may be nothing to show. These components provide that feedback without taking over the operation itself. The application owns the state; the markup describes it.

In [ ]:
from fasthtml.common import Div, Span, H3, P, Label, Template, unqid
from fastcore.xml import FT
from fasthtml.core import FtResponse
from answerui.registry import reg, hypelt
from answerui.controls import Button
from answerui.surfaces import Card
from answerui.layout import Stack
from answerui.demo import example
import answerui.base  # the reset and tokens this styling is written on

In [ ]:
from fasthtml.common import Form
from answerui.demo import demo
from fastcore.test import *

## Notice

An alert and a toast are both a notice: a stack of content beside an optional dismiss button. The stack takes the width and the button stays small. Each adds a surface of its own to this row:


In [ ]:
reg.rule('notice', r"""
:is([alert], [toast]) { display: flex; align-items: flex-start; gap: var(--space); }
:is([alert], [toast]) > [stack] { flex: 1; min-inline-size: 0; }
:is([alert], [toast]) > button[dismiss] { flex-shrink: 0; --pad-block: var(--space-3); --pad-inline: var(--space-2); }
""", layer='components', doc='The row shared by alerts and toasts: the message beside its dismiss button.')

def _notice(c, dismissible, **attrs):
    "A stack of content beside an optional dismiss button"
    close = Button('Dismiss', type='button', secondary=True, dismiss=True) if dismissible else None
    return Div(Stack(*c, gap='var(--space-2)'), close, **attrs)

## Alert

An alert groups a message and any supporting content. It uses the card boundary with a soft accent treatment, coloured by the same severity attributes as badges. Dismissal is optional. It removes the whole message, not just its text, and the delegated handler also works on alerts inserted by htmx. It is a named rule of the alert, so an application that would rather hide the message replaces it by name.

In [ ]:
_alert_css = r"""
[alert] { background: var(--v-3); color: var(--v2); border-color: var(--v-2); }
"""

@hypelt(sel='[alert]', layer='components', css=_alert_css)
def Alert(
    *c,                # The message and any supporting content
    dismissible=False, # Include a button that removes this alert
    **attrs,           # Attributes for the containing div, including severity and role
):
    "An in-page message with optional dismissal"
    return _notice(c, dismissible, card=True, alert=True, **attrs)

Alert.rule('dismiss', on={('& > button[dismiss]', 'click'): "el.closest('[alert]').remove()"}, layer='components', doc='The dismiss button removes the alert.')

Alert.dismiss [components] [alert]: The dismiss button removes the alert.

Colour and announcement are separate decisions. A persistent warning need not interrupt a screen-reader user, so no role is added by default. For polite feedback, establish an empty `role="status"` region in the page, then insert or update the message inside it. Reserve `role="alert"` for urgent messages. Creating a populated status region in one swap is not a reliable way to announce it; see [ARIA live regions](https://developer.mozilla.org/en-US/docs/Web/Accessibility/ARIA/Guides/Live_regions).

In [ ]:
@example(Alert)
def _alert_example():
    return Stack(
        Alert('Your changes have been saved.', success=True, dismissible=True),
        Alert(H3('Storage is nearly full'), P('Remove unused files or choose a larger plan.'), warning=True),
        Alert('The upload failed. Your original file has not changed.', danger=True, dismissible=True))

In [ ]:
Alert.demo()

```html
<div stack>
  <div card alert success>
    <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" secondary dismiss>Dismiss</button>  </div>
  <div card alert warning>
    <div stack style="--gap: var(--space-2);">
      <h3>Storage is nearly full</h3>
      <p>Remove unused files or choose a larger plan.</p>
    </div>
  </div>
  <div card alert danger>
    <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" secondary dismiss>Dismiss</button>  </div>
</div>

```

Persistent alerts have no timer: important information should not disappear before someone has read it.

## Toast

Place an empty `Toasts()` region in the page before delivering messages to it. It is a polite live region in a manual popover: above ordinary page content, without a z-index contest or light dismiss. It does not override a modal opened above it. Feedback needed while a modal is active belongs inside that dialog.

`Toast` goes after five unattended seconds. The lifetime is a CSS animation whose duration is `--life`, set from `life`, and whose last frame hides the toast, so hover or keyboard focus pauses it and leaving resumes the remaining time, with no script keeping time. A hidden toast stays in the region's markup. `life=None` sets no `--life`, and the message stays until dismissed. The reduced-motion rule zeroes `--motion`, which controls only the cosmetic entry transition, and leaves the lifetime alone. The dismissal is a named rule of the toast, so it can be replaced by name.

In [ ]:
_toasts_on = {('&', 'init'): "if (!el.matches(':popover-open')) el.showPopover()"}
_toasts_css = r"""
[toasts][popover] {
  position: fixed; inset: auto; inset-block-start: var(--space); inset-inline-end: var(--space); margin: 0;
  padding: var(--space-3); border: 0; background: none; box-shadow: none; overflow: visible; color: inherit; pointer-events: none;
  inline-size: min(22rem, calc(100vw - 2 * var(--space)));
}
[toasts][popover]:popover-open { display: grid; gap: var(--space-2); }
[toasts] > [toast] { pointer-events: auto; }
"""

@hypelt(sel='[toasts]', layer='components', on=_toasts_on, css=_toasts_css)
def Toasts(*c, **attrs):
    "The page's polite live region for toast deliveries"
    return Div(*c, popover='manual', role='status', aria_live='polite', aria_atomic='false',
        aria_relevant='additions text', toasts=True, **attrs)

In [ ]:
_toast_css = r"""
[toast] {
  --face: var(--n-2); background: var(--face); color: var(--n2); border: 1px solid var(--n-1);
  border-radius: var(--radius); padding: var(--space-2) var(--space); box-shadow: var(--floating);
  transition: opacity var(--motion) var(--ease), translate var(--motion) var(--ease);
  @starting-style { opacity: 0; translate: 0 calc(-1 * var(--space-2)); }
  animation: aui-toast-life var(--life) linear forwards; }
[toast]:is(:hover, :focus-within) { animation-play-state: paused; }
/* The lifetime is a timer that ends by hiding the toast. Without --life the shorthand is invalid, and the toast has no animation. */
@keyframes aui-toast-life { to { display: none; } }
[toast]:is([success], [warning], [danger], [info]) { --face: var(--v-3); color: var(--v2); border-color: var(--v-2); }
"""

@hypelt(sel='[toast]', layer='components', css=_toast_css)
def Toast(
    *c,               # The message
    life=5000,        # Unattended milliseconds before it hides; None to stay until dismissed
    dismissible=True, # Include a dismiss button
    oob=False,        # Wrap for out-of-band delivery into the page's region, beside a fragment
    **attrs,          # Attributes for the message, including severity
):
    "A brief message for the toast region"
    if life: attrs['style'] = f'--life: {life}ms; ' + attrs.get('style', '')
    toast = _notice(c, dismissible, toast=True, **attrs)
    return Div(toast, hx_swap_oob='beforeend:[toasts]') if oob else toast

Toast.rule('dismiss', on={('& > button[dismiss]', 'click'): "el.closest('[toast]').remove()"}, layer='components', doc='The dismiss button removes the toast at once.')

Toast.dismiss [components] [toast]: The dismiss button removes the toast at once.

Deliver a toast by putting it **inside** an out-of-band wrapper: `Div(Toast('Saved'), hx_swap_oob='beforeend:[toasts]')`, which `Toast('Saved', oob=True)` builds. htmx strips the wrapper for `beforeend`, leaving the complete toast, including its attributes and dismiss button. Putting `hx_swap_oob` on the toast itself would strip the component instead. See [out-of-band swaps](https://htmx.org/attributes/hx-swap-oob/).

This example keeps the response in a template and a click on its button passes it to [htmx.swap](https://htmx.org/api/#swap), so it exercises that delivery without a server and registers nothing. An app returns the same wrapper from its route and needs no script of its own. Click to deliver both messages, then try hovering or focusing the timed one.

In [ ]:
@example(Toast)
def _toast_example():
    id = unqid()
    response = Div(Toast('Your changes have been saved.', success=True),
        Toast('The upload failed. Your original file has not changed.', life=None, danger=True),
        hx_swap_oob='beforeend:[toasts]')
    return Div(Button('Deliver notifications', type='button', secondary=True,
        hx_on_click=f"htmx.swap(this, document.getElementById('{id}').innerHTML, {{swapStyle: 'none'}})"),
        Template(response, id=id), Toasts())

In [ ]:
Toast.demo(height='14rem')

```html
<div>
<button type="button" secondary hx-on-click="htmx.swap(this, document.getElementById('_9G7bpjoIRsSH-AT3nejFVw').innerHTML, {swapStyle: 'none'})">Deliver notifications</button><template id="_9G7bpjoIRsSH-AT3nejFVw">    <div hx-swap-oob="beforeend:[toasts]">
      <div toast success style="--life: 5000ms; ">
        <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" secondary dismiss>Dismiss</button>      </div>
      <div toast danger>
        <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" secondary dismiss>Dismiss</button>      </div>
    </div>
</template>  <div popover="manual" role="status" aria-live="polite" aria-atomic="false" aria-relevant="additions text" toasts></div>
</div>

```

### Queued toasts

A route that redirects has no markup to carry a toast, and a route that answers an htmx request wants its toast delivered beside the fragment. Both queue the message in the session with `add_toast`, naming its severity as an attribute. Whatever renders next takes the queue. A full page renders it in its region, `Toasts(*pop_toasts(sess))`. A fragment response delivers `toasts_oob(sess)` beside its content, and htmx swaps the toasts into the region already on screen. `setup_toasts` installs that second delivery as an `after` hook on a FastHTML app, so a route that returns markup need not think about it; a page that has rendered its own region has already taken the queue, and the hook finds nothing.

In [ ]:
toasts_key = 'toasts'

def add_toast(sess, msg, life=5000, **attrs):
    "Queue `msg` in `sess` for the next page or fragment, with its severity in `attrs`"
    sess.setdefault(toasts_key, []).append((msg, life, attrs))

def pop_toasts(sess):
    "The queued toasts as `Toast`s, clearing the queue"
    return [Toast(m, life=l, **a) for m,l,a in sess.pop(toasts_key, [])]

def toasts_oob(sess):
    "The queued toasts in an out-of-band wrapper, for delivery beside a fragment; None when nothing is queued"
    toasts = pop_toasts(sess)
    return Div(*toasts, hx_swap_oob='beforeend:[toasts]') if toasts else None

The hook delivers to a response that is markup, or none at all, which is what a route that only queues a toast returns. A redirect or a file has nowhere to carry them, so they wait for the next page:

In [ ]:
def toast_after(resp, req, sess):
    "FastHTML `after` hook: a markup response, or none, carries the queued toasts out of band"
    if sess.get(toasts_key) and (not resp or isinstance(resp, (tuple, FT, FtResponse))): req.injects.append(toasts_oob(sess))

def setup_toasts(app):
    "Deliver queued toasts beside every markup response of `app`"
    app.after.append(toast_after)

Two messages queued by a route, taken as the out-of-band wrapper a fragment response would carry. Taking them clears the queue:

In [ ]:
sess = {}
add_toast(sess, 'Your changes have been saved.', success=True)
add_toast(sess, 'The upload failed. Your original file has not changed.', life=None, danger=True)
oob = toasts_oob(sess)
test_eq(sess, {})
test_eq([t.attrs.get('danger') for t in oob.children], [None, True])
oob

<div hx-swap-oob="beforeend:[toasts]"><div toast success style="--life: 5000ms; "><div stack style="--gap: var(--space-2);">Your changes have been saved.</div><button type="button" secondary dismiss>Dismiss</button></div><div toast danger><div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div><button type="button" secondary dismiss>Dismiss</button></div></div>

## Progress

When the amount of work is known, a native `progress` element represents the fraction completed. Omitting `value` makes it indeterminate. We keep the browser’s rendering and animation, setting only its dimensions and accent colour. The surrounding label gives the operation a name.

In [ ]:
Progress = hypelt('Progress', layer='components', doc='Native progress; omit value when the amount completed is unknown.', css=r"""
progress { inline-size: 100%; block-size: .75rem; vertical-align: middle; accent-color: var(--v); }
""")

A labelled bar for a known amount of work, and an indeterminate one for an unknown amount:


In [ ]:
@example(Progress)
def _progress_example():
    return Stack(Label('Importing files', Progress(value=40, max=100), stack=True, style='--gap: var(--space-3)'),
        Label('Waiting for the server', Progress(), stack=True, style='--gap: var(--space-3)'))


In [ ]:
Progress.demo()

```html
<div stack>
<label stack style="--gap: var(--space-3)">Importing files<progress value="40" max="100"></progress></label><label stack style="--gap: var(--space-3)">Waiting for the server<progress></progress></label></div>

```

## Spinner

For a short wait, a spinner beside a few words is often enough. It uses the same rotation as busy buttons and follows the motion token. With reduced motion, the ring stays still and the text continues to explain the wait. As with alerts, add a live-region role only where an announcement is wanted.

In [ ]:
@hypelt(sel='[spinner]', layer='components', css=r"""
[spinner] { display: inline-flex; align-items: center; gap: var(--space-2); }
[spinner]::before {
  content: ''; flex-shrink: 0; inline-size: 1em; block-size: 1em;
  border: 2px solid currentColor; border-inline-end-color: transparent; border-radius: var(--radius-full);
  animation: aui-spin calc(var(--motion) * 5) linear infinite;
}
""")
def Spinner(label='Loading…', **attrs):
    "A small spinning ring with visible text explaining the wait"
    return Span(label, spinner=True, **attrs)

In [ ]:
@example(Spinner)
def _spinner_example():
    return Spinner('Checking the connection…')

In [ ]:
Spinner.demo()

```html
<span spinner>Checking the connection…</span>
```

## Skeleton

A skeleton reserves space for content that has not arrived. It is decorative, not a loading message, and stays out of the accessibility tree. Its default shape is a line of text; ordinary CSS sets other dimensions. We leave it still rather than adding another animation.

In [ ]:
@hypelt(sel='[skeleton]', layer='components', css=r"""
[skeleton] { display: block; inline-size: 100%; block-size: 1em;
  background: var(--n-1); border-radius: var(--radius-1); }
""")
def Skeleton(**attrs):
    "A decorative placeholder for content that has not loaded"
    return Span(skeleton=True, aria_hidden='true', **attrs)

In [ ]:
@example(Skeleton)
def _skeleton_example():
    return Stack(P('Loading the project summary…'),
        Card(Skeleton(style='inline-size: 40%; block-size: 1.5em'), Skeleton(), Skeleton(style='inline-size: 75%'),
            aria_busy='true'))

In [ ]:
Skeleton.demo()

```html
<div stack>
  <p>Loading the project summary…</p>
  <div stack card aria-busy="true">
<span skeleton aria-hidden="true" style="inline-size: 40%; block-size: 1.5em"></span><span skeleton aria-hidden="true"></span><span skeleton aria-hidden="true" style="inline-size: 75%"></span>  </div>
</div>

```

The loading message sits outside the busy region. When the content arrives, replace the placeholders and clear `aria-busy`, or replace the whole card. The component does not schedule that work.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()